# Validation v2: frozen CalibratedTotalRisk

This notebook evaluates the Sprint 1 primary revised model without model selection or temporal refitting. It verifies the three canonical SHA256 identities before analysis. `CalibratedTotalRisk` uses the locked training-only coefficients `beta_Demo=1.0586578127615558` and `beta_Physio=0.3563514638575306`.

Historical submitted values remain separately labeled. Cox fits use explicit step-size/iteration/precision controls, and convergence warnings are captured locally rather than suppressed. Bootstrap seed: **20260821**.

Uno's IPCW C and cumulative/dynamic AUC use `scikit-survival` 0.28.0, with training data estimating the censoring distribution and temporal data used for evaluation. Fixed-horizon IPCW Brier scores use the temporal censoring distribution as an evaluation metric.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))
import run_sprint2_validation as s2
print({'bootstrap_seed': s2.SEED, 'bootstrap_replicates': s2.N_BOOT, 'data_dir': str(s2.DATA_DIR)})

The exported validation table contains discrimination, uncertainty, HR per actual test-cohort SD, HR confidence intervals, and age correlation. Paired resampling uses identical bootstrap participants for both scores in each contrast.

In [ ]:
validation = s2.run_validation()
display(pd.read_csv(ROOT/'tables/validation_v2.csv'))
display(pd.read_csv(ROOT/'tables/cindex_bootstrap_contrasts.csv'))

PH diagnostics are descriptive. A flagged result is recorded, not used to alter the model.

In [ ]:
display(pd.read_csv(ROOT/'tables/ph_diagnostics.csv'))

Absolute risk uses the training Cox baseline survival and frozen training coefficients. Temporal outcomes are used only for evaluation. The eight-year horizon is retained with an explicit less-precise-tail qualification.

In [ ]:
display(pd.read_csv(ROOT/'tables/followup_summary.csv'))
display(pd.read_csv(ROOT/'tables/calibration_fixed_horizons.csv'))

Calibration plot data are exported as temporal-test risk deciles; plotting can be styled later without rerunning the model.

In [ ]:
display(pd.read_csv(ROOT/'tables/calibration_plot_data.csv'))

Scaled Schoenfeld residuals and an extended-Cox score × log(time/60 months) model localize non-proportionality. These are diagnostic only. Horizon-specific IPCW logistic calibration slopes are kept distinct from the global PH-dependent Cox slope.